In [ ]:
from pymatgen.core import Composition
import numpy as np
from matminer.featurizers.composition import ElementFraction


class MaxTemperatureSegmentFeaturizer:
    """Featurizer that encodes the maximum synthesis temperature into discrete 50°C segments."""
    def __init__(self, temp_column="max_temp", min_temp=100, max_temp=600, bin_width=50):
        self.temp_column = temp_column
        self.min_temp = min_temp
        self.max_temp = max_temp
        self.bin_width = bin_width
        # Define segment bins (edges) and labels
        self.bins = list(range(min_temp, max_temp + bin_width, bin_width))
        self.labels = [
            f"max_temp_{self.bins[i]}_{self.bins[i+1]}"
            for i in range(len(self.bins) - 1)
        ]

    def featurize(self, composition, **kwargs):
        """
        One-hot encode the temperature segment based on the temperature in either:
        - kwargs["max_temp"]
        - Alternatively, if composition is a dict or pd.Series and self.temp_column is available, use it.
        """
        # Try to extract the temperature value
        temp = None
        if self.temp_column in kwargs:
            temp = kwargs[self.temp_column]
        elif hasattr(composition, "get") and self.temp_column in composition:
            temp = composition[self.temp_column]
        elif hasattr(composition, self.temp_column):
            temp = getattr(composition, self.temp_column)
        else:
            # Fallback, unable to determine temp
            # Return all zeros (out of segment range)
            return [0] * len(self.labels)
        
        # Determine segment
        # Note: If temp is nan or invalid, encode as "all zeros"
        try:
            temp = float(temp)
        except Exception:
            return [0] * len(self.labels)
        encoded = [0] * len(self.labels)
        for i in range(len(self.bins)-1):
            if self.bins[i] <= temp < self.bins[i+1]:
                encoded[i] = 1
                break
        else:
            # If temperature is exactly at last edge, place in the last bin
            if temp == self.bins[-1]:
                encoded[-1] = 1
        return encoded

    def feature_labels(self):
        return self.labels

    def citations(self):
        return []

    def implementors(self):
        return ["Your Name"]


element_featurizer = ElementFraction()
max_temperature_featurizer = MaxTemperatureSegmentFeaturizer()

def featurize(*args) -> np.ndarray:
    composition, max_temperature = args
    elem_feat = element_featurizer.featurize(Composition(composition))
    temp_feat = max_temperature_featurizer.featurize(max_temperature)
    return np.array(elem_feat + temp_feat)

In [ ]:
import pandas as pd
import json
import pymongo
from bson import ObjectId
from datetime import datetime
from typing import Tuple

client = pymongo.MongoClient("mongodb://aragorn:27021/")
db = client["Alab_GPSS"]
collection = db["samples"]
results = json.load(open("../data/dataset.json"))

In [ ]:
import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll_torch
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel
from sklearn.decomposition import PCA
import hashlib
predictions = []
cached_models = {}

import numpy as np
import random

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
# If CUDA is available, set its seed as well for completeness (safe even if not used)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)


for sample in sorted(results, key=lambda x: ObjectId(x["sample_id"]).generation_time):
    sample_id = sample["sample_id"]
    sample_doc = collection.find_one({"_id": ObjectId(sample_id)})
    if "previous_experiment" not in sample_doc["metadata"]:
        continue

    prev_experiment_hash = hashlib.sha256(sample_doc["metadata"]["previous_experiment"].encode()).hexdigest()

    if prev_experiment_hash not in cached_models:
        input_experiments = json.loads(sample_doc["metadata"]["previous_experiment"])
        experiment_df = pd.DataFrame(input_experiments)
        X = np.stack([featurize(row["composition"], row["synthesis_temperature"]) for row in input_experiments])
        y = np.log10(experiment_df["ionic_conductivity_room_temperature (S/cm)"].values)

        # Dimensionality reduction with PCA for compatibility with original workflow.
        pca = PCA(n_components=0.95)
        pca.fit(X)
        X_reduced = pca.transform(X)
        # Min-max scaling of X_reduced to [0, 1] per feature
        X_min = X_reduced.min(axis=0)
        X_max = X_reduced.max(axis=0)
        # To avoid division by zero in case any feature is constant
        X_range = np.where(X_max - X_min == 0, 1, X_max - X_min)
        X_reduced = (X_reduced - X_min) / X_range
        
        # Convert to torch tensors
        train_X = torch.tensor(X_reduced, dtype=torch.float64)
        train_Y = torch.tensor(y.reshape(-1, 1), dtype=torch.float64)

        # Create and fit GP model using BoTorch
        gp = SingleTaskGP(train_X, train_Y, covar_module=MaternKernel())
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll_torch(mll)
        cached_models[prev_experiment_hash] = gp
    else:
        gp = cached_models[prev_experiment_hash]

    target_composition = sample["composition"]
    target_X = np.array(featurize(target_composition, sample["synthesis_temperature"]))
    target_X_reduced = pca.transform(target_X.reshape(1, -1))
    target_X_reduced = (target_X_reduced - X_min) / X_range
    test_X = torch.tensor(target_X_reduced, dtype=torch.float64)

    # Predict using the trained BoTorch GP model
    gp.eval()
    with torch.no_grad(), torch.amp.autocast(device_type="cpu", enabled=False):
        posterior = gp.posterior(test_X)
        y_pred = posterior.mean.item()
        y_std = posterior.variance.sqrt().item()

    print(y_pred, y_std, np.log10(sample["ionic_conductivity_room_temperature (S/cm)"]))
    predictions.append({
        "sample_id": sample_id,
        "prediction": y_pred,
        "std": y_std,
        "actual": np.log10(sample["ionic_conductivity_room_temperature (S/cm)"]),
    })


In [ ]:
from typing import Any


from scipy.stats import norm

for pred in predictions:
    normal_dist = norm(pred["prediction"], pred["std"])
    logprob = normal_dist.logpdf(pred["actual"])
    pred["-logprob"] = -logprob 

prov_all_values = {}

for prov in np.unique([pred["provenance"] for pred in predictions]):
    idx = [pred for pred in predictions if pred["provenance"] == prov]
    all_pred = np.array([pred["-logprob"] for pred in idx])
    print(all_pred[all_pred > 10])
    prov_all_values[prov] = np.array([pred["-logprob"] for pred in idx])

# Violin plot of the distributions
import seaborn as sns
prov_color = {"human": "tab:blue", "abnormal": "tab:red", "novelty": "tab:green", "novelty (BO)": "tab:orange"}
prov_order = ["human", "abnormal", "novelty", "novelty (BO)"]
LABELS = {
    "human": "Initial test",
    "abnormal": "Abnormal",
    "novelty": "Pattern",
    "novelty (BO)": "BO-assisted",
}
plt.rcParams["font.size"] = 14
plt.rcParams["font.family"] = "Arial"
fig, ax = plt.subplots(figsize=(4, 3))

prov_keys = list[Any](prov_all_values.keys())
prov_values = [prov_all_values[k] for k in prov_keys]
# Clip all values > 10 to 10
prov_values = [np.clip(vals, None, 10) for vals in prov_values]

# Compute mean surprise and annotate each class on the plot
for i, k in enumerate(prov_keys):
    mean_val = np.mean(np.clip(prov_all_values[k], None, 10))
    if i < 2:
        ax.text(i+0.5, mean_val+2, f"$\\mu$ = {mean_val:.2f}", ha='center', va='bottom', color=prov_color.get(k, "black"), fontsize=12, fontweight='bold')
    else:
        ax.text(i+0., mean_val-3.5, f"$\\mu$ = {mean_val:.2f}", ha='center', va='bottom', color=prov_color.get(k, "black"), fontsize=12, fontweight='bold')

sns.violinplot(data=prov_values, ax=ax, bw_adjust=0.4, palette=[prov_color[k] for k in prov_keys], alpha=0.8, linewidth=0.75, linecolor="black", saturation=1, inner="box", inner_kws={"box_width": 8, "whis_width": 2, "color": "black", "alpha": 0.8})
ax.set_xticks(range(len(prov_keys)))
ax.set_xticklabels([LABELS[str(k)] for k in prov_keys], rotation=0)
ax.set_ylabel("Shannon surprise")

fig.savefig("bayesian_surprise_by_provenance.pdf", bbox_inches="tight", pad_inches=0.01)

